# Calculadora de metas nutricionais diárias

O código nesse *notebook* puxa informações nutricionais de múltiplas fontes para cálculo da quantidade de elementos nutricionais diários listados pela TACO para uma alimentação saudável. Após computar os dados nutricionais necessários, eles serão inseridos num algoritmo de otimização com programação linear, para qualquer uma das características dos alimentos presentes.

- Fontes:
    - UNICAMP/NEPA - Tabela Brasileira de Composição de Alimentos (TACO);
    - NASEM/Health Canada - Dietary Reference Intakes, equations to estimate energy requirement;
    - Health Canada / Food and Nutrition Board - Dietary Reference Intakes tables;
    - National Academies 2019 - Dietary Reference Intakes for Sodium and Potassium;
    - WHO/FAO/UNU 2007 and FAO 2011 protein quality consultation tables.

In [26]:
import csv
from pathlib import Path
import pandas as pd

from functions import (
    calcular_necessidades,
    formatar_numero_brasileiro,
    formatar_numero_exportacao,
    limpar_rotulo,
    otimizar_dieta_lp,
    formatar_quantidade,
    formatar_tabela_exportacao,
    preparar_cobertura_humana,
    preparar_plano_humano,
    preparar_resumo_categorias,
)

from IPython.display import Markdown, display  # pyright: ignore[reportUnknownVariableType]

In [27]:
DATA_DIR = Path("../data")
ID_COL = "Número do Alimento"

alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")

aminoacidos["Triptofano (g)"] = pd.to_numeric(
    aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

taco_completo = alimentos.merge(
    acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

output_path = DATA_DIR / "taco_completo.csv"
taco_completo.to_csv(output_path, index=False, na_rep="NA")

taco_completo.head()

try:
    tabela_bruta = taco_completo.copy()
except NameError:
    alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
    acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
    aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")
    aminoacidos["Triptofano (g)"] = pd.to_numeric(
        aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    tabela_bruta = alimentos.merge(
        acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
    ).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

colunas_repetidas = [
    coluna
    for coluna in tabela_bruta.columns
    if coluna.endswith(("_acidos_graxos", "_aminoacidos"))
    and coluna.startswith(("Categoria do alimento", "Descrição dos alimentos"))
]
tabela_limpa = tabela_bruta.drop(columns=colunas_repetidas)

tabela_formatada = tabela_limpa.copy()
colunas_numericas = tabela_formatada.select_dtypes(include="number").columns

for coluna in colunas_numericas:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(formatar_numero_brasileiro)

for coluna in tabela_formatada.columns.difference(colunas_numericas):
    tabela_formatada[coluna] = tabela_formatada[coluna].replace("NA", pd.NA).fillna("")

tabela_formatada.columns = [
    limpar_rotulo(coluna) for coluna in tabela_formatada.columns
]

output_path = DATA_DIR / "taco_completo.csv"
tabela_formatada.to_csv(output_path, sep=";", index=False, quoting=csv.QUOTE_ALL)

tabela_formatada.head()

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Umidade,Energia (kcal),Energia (kJ),Proteína (g),Lipídeos (g),Colesterol (mg),Carboidrato (g),...,Tirosina (g),Valina (g),Arginina (g),Histidina (g),Alanina (g),Ácido Aspártico (g),Ácido Glutâmico (g),Glicina (g),Prolina (g),Serina (g)
0,1,Cereais e derivados,"Arroz, integral, cozido","70,1",124,517,"2,6",1,,"25,8",...,,,,,,,,,,
1,2,Cereais e derivados,"Arroz, integral, cru","12,2",360,1505,"7,3","1,9",,"77,5",...,,,,,,,,,,
2,3,Cereais e derivados,"Arroz, tipo 1, cozido","69,1",128,537,"2,5","0,2",,"28,1",...,,,,,,,,,,
3,4,Cereais e derivados,"Arroz, tipo 1, cru","13,2",358,1497,"7,2","0,3",,"78,8",...,,,,,,,,,,
4,5,Cereais e derivados,"Arroz, tipo 2, cozido","68,7",130,544,"2,6","0,4",,"28,2",...,,,,,,,,,,


In [28]:
necessidades_nutricionais = calcular_necessidades(
    sexo="masculino",
    idade_anos=25,
    altura_m=1.84,
    peso_kg=136,
    colunas_taco=list(tabela_formatada.columns)
)

necessidades_exportacao = necessidades_nutricionais.copy()
for coluna in ["EER usado (kcal/dia)", "Alvo", "Mínimo", "Máximo"]:
    necessidades_exportacao[coluna] = necessidades_exportacao[coluna].map(
        formatar_numero_exportacao
    )

saida_necessidades = Path("../data") / "necessidades_nutricionais_estimadas.csv"
necessidades_exportacao.to_csv(
    saida_necessidades, sep=";", index=False, quoting=csv.QUOTE_ALL
)

necessidades_nutricionais

,Estágio de vida,EER usado (kcal/dia),Nutriente,Colunas TACO usadas,Tipo,Alvo,Mínimo,Máximo,Unidade,Base científica,Observações
0,male_19_30,4098,Energia,Energia (kcal),EER,4098.0,NaN,NaN,kcal/dia,"Equação NASEM 2023 por sexo, idade, altura, pe...",
1,male_19_30,4098,Carboidrato,Carboidrato (g),RDA + AMDR,130.0,461.0,665.8,g/dia,DRI: RDA e 45-65% da energia,
2,male_19_30,4098,Proteína,Proteína (g),RDA por kg + AMDR,108.8,102.4,358.5,g/dia,"0.8 g/kg/dia, com mínimo de referência 56 g/dia",
3,male_19_30,4098,Lipídeos totais,Lipídeos (g),AMDR,NaN,91.1,159.3,g/dia,Percentual de energia vindo de gorduras totais,
4,male_19_30,4098,Fibra Alimentar,Fibra Alimentar (g),AI estimada por energia,57.4,NaN,NaN,g/dia,14 g/1000 kcal; tabela DRI também informa AI p...,AI do estágio de vida na tabela: 38 g/dia
...,...,...,...,...,...,...,...,...,...,...,...
56,male_19_30,4098,Ácido Aspártico (g),Ácido Aspártico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
57,male_19_30,4098,Ácido Glutâmico (g),Ácido Glutâmico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
58,male_19_30,4098,Glicina (g),Glicina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
59,male_19_30,4098,Prolina (g),Prolina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."


In [29]:
display(
    pd.DataFrame(
        tabela_formatada["Categoria do Alimento"].unique(), columns=["Unique Values"]
    )
)

,Unique Values
0,Cereais e derivados
1,"Verduras, hortaliças e derivados"
2,Frutas e derivados
3,Gorduras e óleos
4,Pescados e frutos do mar
5,Carnes e derivados
6,Leite e derivados
7,Bebidas (alcoólicas e não alcoólicas)
8,Ovos e derivados
9,Produtos açucarados


In [30]:
# Linear programming model:
# x[i] = number of 100 g portions of food i.
# Objective: minimize total grams of food while meeting the computed nutrient bounds.
MAX_GRAMAS_POR_ALIMENTO = 500
TOLERANCIA_ENERGIA_ACIMA = 0.05
MIN_GRAMAS_PARA_EXIBIR = 0.1
CATEGORIAS_PERMITIDAS = [
    "Cereais e derivados",
    "Verduras, hortaliças e derivados",
    "Frutas e derivados",
    "Gorduras e óleos",
    "Pescados e frutos do mar",
    "Carnes e derivados",
    "Ovos e derivados",
    "Miscelâneas",
    "Leguminosas e derivados",
    "Nozes e sementes",
]  # Example: ["Cereais e derivados", "Carnes e derivados"]
TERMOS_EXCLUIDOS = None  # Example: ["café, pó", "gelatina", "fermento"]

COLUNAS_IDENTIFICACAO = [
    "Número do Alimento",
    "Categoria do Alimento",
    "Descrição dos Alimentos",
]

try:
    tabela_base_lp = tabela_limpa.copy()
except NameError:
    tabela_base_lp = pd.read_csv(DATA_DIR / "taco_completo.csv", sep=";")

try:
    necessidades_lp = necessidades_nutricionais.copy()
except NameError:
    necessidades_lp = pd.read_csv(
        DATA_DIR / "necessidades_nutricionais_estimadas.csv", sep=";"
    )

resumo_otimizacao, dieta_otimizada, cobertura_dieta_otimizada = otimizar_dieta_lp(
    tabela_base=tabela_base_lp,
    necessidades=necessidades_lp,
    max_gramas_por_alimento=MAX_GRAMAS_POR_ALIMENTO,
    tolerancia_energia_acima=TOLERANCIA_ENERGIA_ACIMA,
    categorias_permitidas=CATEGORIAS_PERMITIDAS,
    termos_excluidos=TERMOS_EXCLUIDOS,
    min_gramas_para_exibir=MIN_GRAMAS_PARA_EXIBIR,
)

for caminho, tabela in [
    (DATA_DIR / "dieta_otimizada_lp.csv", dieta_otimizada),
    (DATA_DIR / "cobertura_dieta_otimizada_lp.csv", cobertura_dieta_otimizada),
]:
    tabela_exportacao = tabela.copy()
    for coluna in tabela_exportacao.select_dtypes(include="number").columns:
        tabela_exportacao[coluna] = tabela_exportacao[coluna].map(
            formatar_numero_exportacao
        )
    tabela_exportacao.to_csv(caminho, sep=";", index=False, quoting=csv.QUOTE_ALL)

dieta_otimizada

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Quantidade (g),Porções de 100 g,Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano
0,580,Leguminosas e derivados,"Pé-de-moleque, amendoim",368.621910,3.686219,1854.168206,48.658092,201.636185,103.214135,12.533145,5.897951e+01
1,489,Ovos e derivados,"Ovo, de galinha, inteiro, cru",192.062385,1.920624,274.649210,24.968110,3.072998,17.093552,0.000000,3.226648e+02
2,323,Carnes e derivados,Apresuntado,183.382130,1.833821,236.562947,24.756588,5.318082,12.286603,0.000000,1.729293e+03
3,27,Cereais e derivados,"Creme de arroz, pó",136.081950,1.360820,525.276328,9.525737,114.172756,1.632983,1.496901,1.360820e+00
4,46,Cereais e derivados,"Mingau tradicional, pó",120.857719,1.208577,450.799291,0.725146,107.925943,0.483431,1.087719,1.812866e+01
5,253,Frutas e derivados,"Tucumã, cru",75.668046,0.756680,198.250281,1.589029,20.052032,14.452597,9.609842,3.026722e+00
6,515,Miscelâneas,"Gelatina, sabores variados, pó",70.230234,0.702302,266.874889,6.250491,62.645369,0.000007,0.000000,1.650411e+02
7,511,Miscelâneas,"Café, pó, torrado",56.855382,0.568554,238.224050,8.357741,37.410841,6.765790,29.109955,5.685538e-01
8,594,Nozes e sementes,"Linhaça, semente",10.404468,0.104045,51.502117,1.467030,4.505135,3.360643,3.485497,9.364021e-01
9,169,Frutas e derivados,"Acerola, crua",5.129336,0.051293,1.692681,0.046164,0.410347,0.010259,0.076940,5.129336e-07


In [31]:
try:
    dieta_base_humana = dieta_otimizada.copy()
except NameError:
    dieta_base_humana = pd.read_csv(DATA_DIR / "dieta_otimizada_lp.csv", sep=";")

try:
    cobertura_base_humana = cobertura_dieta_otimizada.copy()
except NameError:
    cobertura_base_humana = pd.read_csv(
        DATA_DIR / "cobertura_dieta_otimizada_lp.csv", sep=";"
    )

plano_alimentar_legivel = preparar_plano_humano(dieta_base_humana)
resumo_por_categoria = preparar_resumo_categorias(plano_alimentar_legivel)
cobertura_legivel = preparar_cobertura_humana(cobertura_base_humana)

arquivos_legiveis = {
    DATA_DIR / "plano_alimentar_legivel.csv": plano_alimentar_legivel,
    DATA_DIR / "plano_alimentar_resumo_categorias.csv": resumo_por_categoria,
    DATA_DIR / "plano_alimentar_cobertura_legivel.csv": cobertura_legivel,
}

for caminho, tabela in arquivos_legiveis.items():
    formatar_tabela_exportacao(tabela).to_csv(
        caminho, sep=";", index=False, quoting=csv.QUOTE_ALL
    )

total_diario = formatar_quantidade(
    plano_alimentar_legivel["Quantidade diária (g)"].sum()
)
total_semanal = formatar_quantidade(
    plano_alimentar_legivel["Quantidade semanal (g)"].sum()
)
restricoes_ok = int(cobertura_legivel["Status"].eq("OK").sum())
total_restricoes = len(cobertura_legivel)

linhas_markdown = [
    "# Plano alimentar otimizado",
    "",
    f"- Total aproximado: {total_diario} por dia",
    f"- Equivalente semanal: {total_semanal} por semana",
    f"- Alimentos no plano: {len(plano_alimentar_legivel)}",
    f"- Restrições nutricionais atendidas: {restricoes_ok}/{total_restricoes}",
    "",
    "## Alimentos",
    "",
]

for _, linha in plano_alimentar_legivel.iterrows():
    observacao = (
        f" ({linha['Observação prática']})" if linha["Observação prática"] else ""
    )
    linhas_markdown.append(
        f"- {linha['Descrição dos Alimentos']}: {linha['Formato sugerido']}{observacao}"
    )

linhas_markdown.extend(
    [
        "",
        "## Nota",
        "",
        (
            "Este plano minimiza massa total de alimentos, não sabor, "
            "variedade, custo, saciedade ou adequação culinária. Use os "
            "campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula "
            "de otimização para deixar o resultado mais parecido com "
            "comida de verdade."
        ),
    ]
)

relatorio_markdown = "\n".join(linhas_markdown)
(DATA_DIR / "plano_alimentar_legivel.md").write_text(
    relatorio_markdown, encoding="utf-8"
)

display(Markdown(relatorio_markdown))
plano_alimentar_legivel

# Plano alimentar otimizado

- Total aproximado: 1220 g por dia
- Equivalente semanal: 8540 g por semana
- Alimentos no plano: 10
- Restrições nutricionais atendidas: 33/33

## Alimentos

- Pé-de-moleque, amendoim: 370 g por dia (porção diária alta)
- Ovo, de galinha, inteiro, cru: 190 g por dia (peso da TACO pode mudar após preparo)
- Apresuntado: 185 g por dia
- Creme de arroz, pó: 135 g por dia (peso da TACO pode mudar após preparo)
- Mingau tradicional, pó: 120 g por dia (peso da TACO pode mudar após preparo)
- Tucumã, cru: 530 g por semana (~76 g/dia) (peso da TACO pode mudar após preparo)
- Gelatina, sabores variados, pó: 490 g por semana (~70 g/dia) (peso da TACO pode mudar após preparo; item denso em açúcar; revisar se quiser um cardápio mais realista)
- Café, pó, torrado: 400 g por semana (~57 g/dia) (peso da TACO pode mudar após preparo)
- Linhaça, semente: 73 g por semana (~10,5 g/dia) (peso da TACO pode mudar após preparo)
- Acerola, crua: 36 g por semana (microquantidade; mais fácil planejar por semana; peso da TACO pode mudar após preparo)

## Nota

Este plano minimiza massa total de alimentos, não sabor, variedade, custo, saciedade ou adequação culinária. Use os campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula de otimização para deixar o resultado mais parecido com comida de verdade.

,Descrição dos Alimentos,Categoria do Alimento,Formato sugerido,Quantidade diária (g),Quantidade semanal (g),Quantidade mensal (g),Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano,Observação prática
0,"Pé-de-moleque, amendoim",Leguminosas e derivados,370 g por dia,370.0,2580,11060,1854.0,48.7,201.6,103.2,12.5,59.0,porção diária alta
1,"Ovo, de galinha, inteiro, cru",Ovos e derivados,190 g por dia,190.0,1345,5760,275.0,25.0,3.1,17.1,0.0,323.0,peso da TACO pode mudar após preparo
2,Apresuntado,Carnes e derivados,185 g por dia,185.0,1285,5500,237.0,24.8,5.3,12.3,0.0,1729.0,
3,"Creme de arroz, pó",Cereais e derivados,135 g por dia,135.0,955,4080,525.0,9.5,114.2,1.6,1.5,1.0,peso da TACO pode mudar após preparo
4,"Mingau tradicional, pó",Cereais e derivados,120 g por dia,120.0,845,3625,451.0,0.7,107.9,0.5,1.1,18.0,peso da TACO pode mudar após preparo
5,"Tucumã, cru",Frutas e derivados,530 g por semana (~76 g/dia),76.0,530,2270,198.0,1.6,20.1,14.5,9.6,3.0,peso da TACO pode mudar após preparo
6,"Gelatina, sabores variados, pó",Miscelâneas,490 g por semana (~70 g/dia),70.0,490,2105,267.0,6.3,62.6,0.0,0.0,165.0,peso da TACO pode mudar após preparo; item den...
7,"Café, pó, torrado",Miscelâneas,400 g por semana (~57 g/dia),57.0,400,1705,238.0,8.4,37.4,6.8,29.1,1.0,peso da TACO pode mudar após preparo
8,"Linhaça, semente",Nozes e sementes,"73 g por semana (~10,5 g/dia)",10.5,73,310,52.0,1.5,4.5,3.4,3.5,1.0,peso da TACO pode mudar após preparo
9,"Acerola, crua",Frutas e derivados,36 g por semana,5.0,36,155,2.0,0.0,0.4,0.0,0.1,0.0,microquantidade; mais fácil planejar por seman...
